## Data Processing in Python Final Project

### Authors: Matyáš Tvrz, 

### To Do List:

1) download data from bezrealitky.cz and reality.idnes.cz, and from sreality.cz on other cities
2) create heatmap based on longitude and latitude - DONE
3) add property popups to map?
4) download info on airbnb prices
5) conduct a simple analysis of rental price determinants

In [11]:
# import packages

import json
import pandas as pd
import os
import requests 
import pandas as pd 
import time
import re 
import random 
import folium
from folium.plugins import HeatMap, HeatMapWithTime


In [12]:
from function_scripts import request_multiple_sreality
df = request_multiple_sreality(1, 10, 'flat', 'rent', locality_region_id=10)

In [13]:
df.shape

(189, 27)

In [14]:
df.head()

,labelsReleased,has_panorama,labels,is_auction,labelsAll,seo,exclusively_at_rk,category,has_floor_plan,_embedded,...,hash_id,attractive_offer,price,price_czk,_links,rus,name,region_tip,gps,has_matterport_url
0,"[[], []]",0,[],False,"[[personal, balcony, brick, elevator], [candy_...","{'category_main_cb': 1, 'category_sub_cb': 4, ...",0,1,1,"{'favourite': {'is_favourite': False, '_links'...",...,2931732556,0,33000,"{'value_raw': 33000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+kk 59 m²,3618187,"{'lat': 50.09975387785671, 'lon': 14.456826122...",False
1,"[[], []]",0,[],False,"[[personal, balcony, brick], [small_shop, thea...","{'category_main_cb': 1, 'category_sub_cb': 5, ...",1,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,3912172364,0,27000,"{'value_raw': 27000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+1 59 m²,0,"{'lat': 50.06304887785671, 'lon': 14.447527122...",False
2,"[[], []]",0,[],False,"[[personal, brick], [theater, small_shop, tave...","{'category_main_cb': 1, 'category_sub_cb': 5, ...",0,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,4280918092,0,27000,"{'value_raw': 27000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+1 63 m²,0,"{'lat': 50.06352987785671, 'lon': 14.447389122...",False
3,"[[partly_furnished], []]",0,[Částečně vybavený],False,"[[personal, balcony, brick, elevator, partly_f...","{'category_main_cb': 1, 'category_sub_cb': 5, ...",1,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,1913217868,0,32000,"{'value_raw': 32000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+1 85 m² (Jednopodlažní),0,"{'lat': 50.06656587785671, 'lon': 14.449175122...",False
4,"[[], []]",0,[],False,"[[personal, brick, cellar, furnished], [vet, c...","{'category_main_cb': 1, 'category_sub_cb': 4, ...",0,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,1669242956,0,27000,"{'value_raw': 27000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+kk 53 m² (Podkrovní),0,"{'lat': 50.09753387785671, 'lon': 14.387732122...",False


In [15]:
columns_to_keep = ['locality', 'price', 'name', 'gps','hash_id','exclusively_at_rk']
df_clean = df[columns_to_keep].copy()
df_clean.head()

,locality,price,name,gps,hash_id,exclusively_at_rk
0,Praha 7 - Holešovice,33000,Pronájem bytu 2+kk 59 m²,"{'lat': 50.09975387785671, 'lon': 14.456826122...",2931732556,0
1,Praha 2 - Vinohrady,27000,Pronájem bytu 2+1 59 m²,"{'lat': 50.06304887785671, 'lon': 14.447527122...",3912172364,1
2,Praha 2 - Vinohrady,27000,Pronájem bytu 2+1 63 m²,"{'lat': 50.06352987785671, 'lon': 14.447389122...",4280918092,0
3,Praha 2 - Vinohrady,32000,Pronájem bytu 2+1 85 m² (Jednopodlažní),"{'lat': 50.06656587785671, 'lon': 14.449175122...",1913217868,1
4,Praha 6 - Dejvice,27000,Pronájem bytu 2+kk 53 m² (Podkrovní),"{'lat': 50.09753387785671, 'lon': 14.387732122...",1669242956,0


In [16]:
df_clean[['lat', 'lon']] = df_clean.gps.apply(lambda x: pd.Series({'lat': x['lat'], 'lon': x['lon']}))
df_clean.head()

,locality,price,name,gps,hash_id,exclusively_at_rk,lat,lon
0,Praha 7 - Holešovice,33000,Pronájem bytu 2+kk 59 m²,"{'lat': 50.09975387785671, 'lon': 14.456826122...",2931732556,0,50.099754,14.456826
1,Praha 2 - Vinohrady,27000,Pronájem bytu 2+1 59 m²,"{'lat': 50.06304887785671, 'lon': 14.447527122...",3912172364,1,50.063049,14.447527
2,Praha 2 - Vinohrady,27000,Pronájem bytu 2+1 63 m²,"{'lat': 50.06352987785671, 'lon': 14.447389122...",4280918092,0,50.063530,14.447389
3,Praha 2 - Vinohrady,32000,Pronájem bytu 2+1 85 m² (Jednopodlažní),"{'lat': 50.06656587785671, 'lon': 14.449175122...",1913217868,1,50.066566,14.449175
4,Praha 6 - Dejvice,27000,Pronájem bytu 2+kk 53 m² (Podkrovní),"{'lat': 50.09753387785671, 'lon': 14.387732122...",1669242956,0,50.097534,14.387732


In [17]:
df_heatmap = df_clean[['lat', 'lon', 'price']].copy()

In [22]:
hm = folium.Map(location=(49.75, 15.40), zoom_start = 8)
HeatMap(df_heatmap, 
        min_opacity=0.4,
        blur = 18
               ).add_to(folium.FeatureGroup(name='Heat Map').add_to(hm))
folium.LayerControl().add_to(m)
hm.save("heatmap.html")